# 01 - Inspect the PDF and compare the source image
Read the original bytes before changing the input. Inspect every page's images, effective resolution and text layer. Only extract an unchanged page image when one complete image represents the page; a background or stencil by itself is not a substitute for the visible page.

This notebook is self-contained. Run from `notebooks/` and configure only [the OCR-local .env](../.env.example). Inputs, expected values, screenshots and responses remain in ignored directories. No private file is embedded in the code.

In [ ]:
import hashlib, json
from pathlib import Path
import pandas as pd
import pikepdf
import pypdfium2 as pdfium
from PIL import Image
from dotenv import dotenv_values

OCR = Path('..').resolve()
if not (OCR / '.env').is_file():
    raise FileNotFoundError('Copy demo/ocr/.env.example to demo/ocr/.env and configure it first.')
config = dotenv_values(OCR / '.env')
PDF = (OCR / config['OCR_INPUT_FILE']).resolve()
SOURCE = (OCR / config['OCR_SOURCE_IMAGE']).resolve() if config.get('OCR_SOURCE_IMAGE') else None
PAGE = int(config.get('OCR_FOCUS_PAGE', 1))
DPI = int(config.get('OCR_TARGET_DPI', 300))
ANALYZER = config.get('CU_ANALYZER_ID') or 'prebuilt-layout'
API = config.get('CU_API_VERSION') or '2025-11-01'
OUT = OCR / 'output'
(OUT / 'versions').mkdir(parents=True, exist_ok=True)
(OUT / 'screenshots').mkdir(exist_ok=True)
original_hash = hashlib.sha256(PDF.read_bytes()).hexdigest()
print('Original bytes:', PDF.stat().st_size, '| SHA256:', original_hash)

## Inspect all page objects, including nested forms
Use [PDFium page objects and image metadata](https://pypdfium2.readthedocs.io/en/stable/python_api.html#pypdfium2.PdfPage.get_objects), not just the first `/Image` resource. Effective DPI is the image resolution at its placed size. A render-mode-3 text object is invisible; this does not establish that a service chose that text over pixels. A page with several masks/images cannot safely be reduced to the first image.

In [ ]:
image_rows, page_rows = [], []
with pdfium.PdfDocument(str(PDF)) as document:
    for number, page in enumerate(document, 1):
        text = page.get_textpage().get_text_range()
        objects = list(page.get_objects())
        images = [obj for obj in objects if obj.type == pdfium.raw.FPDF_PAGEOBJ_IMAGE]
        hidden = any(pdfium.raw.FPDFTextObj_GetTextRenderMode(obj) == 3
                     for obj in objects if obj.type == pdfium.raw.FPDF_PAGEOBJ_TEXT)
        page_rows.append({'page': number, 'text_characters': len(text), 'invisible_text': hidden, 'image_objects': len(images)})
        (OUT / f'page_{number}_embedded_text.txt').write_text(text, encoding='utf-8')
        for index, obj in enumerate(images):
            meta = obj.get_metadata()
            xdpi, ydpi = meta.horizontal_dpi, meta.vertical_dpi
            mode = 'standard' if abs(xdpi - 204) < 10 and abs(ydpi - 98) < 10 else 'other'
            if abs(xdpi - 204) < 10 and abs(ydpi - 196) < 10:
                mode = 'fine'
            image_rows.append({'page': number, 'image': index + 1, 'pixels': f'{meta.width} x {meta.height}',
                               'bits_per_pixel': meta.bits_per_pixel, 'filters': ', '.join(obj.get_filters()),
                               'x_dpi': round(xdpi, 1), 'y_dpi': round(ydpi, 1), 'fax_mode': mode})
display(pd.DataFrame(page_rows))
display(pd.DataFrame(image_rows))
pd.DataFrame(page_rows).to_csv(OUT / 'page_structure.csv', index=False)
pd.DataFrame(image_rows).to_csv(OUT / 'image_structure.csv', index=False)

## Original intake image or conservative embedded-image extraction
An intake image can be configured explicitly. Otherwise this cell uses [pikepdf.PdfImage.as_pil_image](https://pikepdf.readthedocs.io/en/latest/topics/images.html) only when every page has exactly one full-page image with no mask. Decoded pixels are saved losslessly, without resampling or changing mode. Original JPEG encoding is not reproduced as a newly written JPEG.

If extraction is unsafe, the source-image case is explicitly unavailable. Full-page rendering in notebook 02 is a different experiment, not an unchanged source image.

In [ ]:
source_file = SOURCE
reason = ''
if source_file is None:
    pages, resolution = [], None
    with pikepdf.open(PDF) as pdf, pdfium.PdfDocument(str(PDF)) as document:
        for number, page in enumerate(pdf.pages):
            visual = document[number]
            page_objects = list(visual.get_objects())
            objects = [obj for obj in page_objects if obj.type == pdfium.raw.FPDF_PAGEOBJ_IMAGE]
            visible_extras = [obj for obj in page_objects if obj.type not in (pdfium.raw.FPDF_PAGEOBJ_IMAGE, pdfium.raw.FPDF_PAGEOBJ_FORM) and not (obj.type == pdfium.raw.FPDF_PAGEOBJ_TEXT and pdfium.raw.FPDFTextObj_GetTextRenderMode(obj) == 3)]
            if visible_extras:
                reason = 'Visible text or vector objects would be lost by image-only extraction; render the page.'
                break
            if len(objects) != 1 or len(page.images) != 1:
                reason = 'Multiple images or stencil layers: no single unchanged full-page source image.'
                break
            image = next(iter(page.images.values()))
            bounds = objects[0].get_bounds()
            width, height = visual.get_size()
            if image.get('/ImageMask', False) or '/SMask' in image or '/Mask' in image:
                reason = 'Masked image: use a full-page render rather than dropping its mask.'
                break
            if abs(bounds[2] - bounds[0] - width) > 2 or abs(bounds[3] - bounds[1] - height) > 2:
                reason = 'The image does not cover the page: use notebook 02.'
                break
            meta = objects[0].get_metadata()
            current = (meta.horizontal_dpi, meta.vertical_dpi)
            if resolution is not None and current != resolution:
                reason = 'Page resolutions differ; do not silently assign one DPI to every source page.'
                break
            resolution = current
            pages.append(pikepdf.PdfImage(image).as_pil_image().copy())
    if not reason and pages:
        source_file = OUT / 'versions' / 'source_image.tiff'
        compression = 'group4' if all(page.mode == '1' for page in pages) else 'tiff_lzw'
        pages[0].save(source_file, save_all=True, append_images=pages[1:], compression=compression, dpi=resolution)
        print('Saved decoded source pixels without resampling:', len(pages), 'pages.')
    for page in pages:
        page.close()
print('Source image:', 'available' if source_file is not None else 'unavailable - ' + reason)
(OUT / 'source_image_status.json').write_text(json.dumps({'available': source_file is not None, 'reason': reason}, indent=2), encoding='utf-8')

## Compare with Content Understanding
The client and request are adapted from Microsoft's [binary-analysis sample](https://github.com/Azure/azure-sdk-for-python/blob/53023a2cd9adf255cff520a99985fff84dc58d7d/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_analyze_binary.py). [MIT notice](../LICENSE). The same GA API and analyzer are used for each case. The local `.env` is the only settings source; Entra authentication is used unless an optional key is explicitly set there.

Responses are cached by file bytes, endpoint, analyzer and API version. Only a cache miss submits a billable request. Missing endpoint configuration is reported as skipped, never as a successful OCR result.

In [ ]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential

endpoint = config.get('CONTENTUNDERSTANDING_ENDPOINT', '').rstrip('/')
key = config.get('CONTENTUNDERSTANDING_KEY')
client = ContentUnderstandingClient(endpoint=endpoint, credential=AzureKeyCredential(key) if key else DefaultAzureCredential(process_timeout=60), api_version=API) if endpoint else None

def analyze(path, label):
    if client is None:
        print('Skipped: set CONTENTUNDERSTANDING_ENDPOINT in demo/ocr/.env')
        return None
    data = path.read_bytes()
    digest = hashlib.sha256(data + f'{endpoint}|{ANALYZER}|{API}'.encode()).hexdigest()
    cache = OUT / 'cu-cache' / f'{digest}.json'
    cache.parent.mkdir(exist_ok=True)
    if cache.exists():
        result = json.loads(cache.read_text(encoding='utf-8'))
        print(label, '- cached')
    else:
        print(label, '- submitting to', ANALYZER)
        result = client.begin_analyze_binary(analyzer_id=ANALYZER, binary_input=data).result().as_dict()
        cache.write_text(json.dumps(result, indent=2), encoding='utf-8')
    text = '\n\n'.join(item.get('markdown', '') for item in result['contents'])
    (OUT / f'{label}.md').write_text(text, encoding='utf-8')
    metadata = {'input_sha256': hashlib.sha256(data).hexdigest(), 'cache': cache.name, 'analyzer': ANALYZER, 'api_version': API}
    (OUT / f'{label}.input.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    return result

responses = {'original_pdf': analyze(PDF, 'original_pdf')}
if source_file is not None:
    responses['source_image'] = analyze(source_file, 'source_image')
else:
    print('Source-image OCR not run:', reason)

## Inspect the visible sign and the OCR context
The screenshot below is a full-page render crop for inspection only. It is not labelled as the original embedded image. Crop coordinates are in inches in the local `.env`; only that crop is displayed. The complete response remains local. A keyword match is not an accuracy score; notebook 04 checks expected amounts against the correct row.

In [ ]:
box = tuple(float(v) for v in config['OCR_CROP_BOX'].split(','))
with pdfium.PdfDocument(str(PDF)) as document:
    bitmap = document[PAGE - 1].render(scale=DPI / 72)
    try:
        image = bitmap.to_pil().convert('L')
    finally:
        bitmap.close()
crop = image.crop(tuple(round(v * DPI) for v in box))
crop.save(OUT / 'screenshots' / 'original_visible_crop.png', dpi=(DPI, DPI))
display(crop)
image.close()
keyword = config.get('OCR_KEYWORD', 'property tax').casefold()
for label, result in responses.items():
    print('\n', label)
    if result is None:
        print('Not analyzed.')
        continue
    lines = '\n'.join(item.get('markdown', '') for item in result['contents']).splitlines()
    found = [i for i, line in enumerate(lines) if keyword in line.casefold()]
    if not found:
        print('Label not found; inspect saved Markdown.')
    for i in found[:2]:
        print('\n'.join(lines[max(0, i - 1):i + 3]))
print('Original unchanged:', hashlib.sha256(PDF.read_bytes()).hexdigest() == original_hash)

## Takeaways
- Establish the actual PDF structure rather than assuming standard fax resolution or invisible text.
- Do not send a JPEG background or one stencil as if it were the complete page.
- If a complete source image is unavailable, continue with explicit full-page rendering in notebook 02.
- A change in OCR between representations narrows the symptom; it does not prove which internal service path caused it.